# ICT Daily Quiz 2028 — Restore missing Google Drive crops

This notebook starts with the committed 17-quiz baseline, reconstructs all 95 quizzes / 950 WEBP images from PDF pairs, verifies marking answers and pixel content, and uploads only missing files to the existing Google Drive folders.

**Instructions:** In Google Colab, select **Runtime → Run all**, then sign in to the Google account that has edit access to the quiz Drive folders. Do not paste an API key or service-account secret. Re-run safely if interrupted; existing nonempty files are preserved. This workflow does not depend on GitHub Actions.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/ict-daily-quiz')
url = 'https://github.com/adnimeshakalhara04-create/ict-daily-quiz.git'
if not (repo / '.git').exists():
    subprocess.run(['git','clone','--depth','1',url,str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pymupdf','pillow','google-api-python-client','google-auth'],check=True)
os.chdir(repo)
print('Original source PDFs and cropping code ready.')

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
service = build('drive','v3',credentials=credentials,cache_discovery=False)
for folder_id in ('1I9fU_ojRQBS-U3_5slnkWGN_SIsgRjmN','1mYROVvho24eDc3ys_OA2C2dmDq5cZ9NL'):
    meta = service.files().get(fileId=folder_id, fields='name,mimeType').execute()
    assert meta['mimeType']=='application/vnd.google-apps.folder'
    print('Drive access confirmed:',meta['name'])

In [ ]:
# Restore generated Quiz 14 repair bytes and build from the committed 17-quiz baseline.
# Never require the committed manifest to already contain 95 quizzes.
import json, sys
from pathlib import Path
from PIL import Image, ImageStat
import fitz

sys.path.insert(0, str(repo / 'scripts'))
import process_incoming as cropper
import process_incoming_auto as auto_cropper

repair = repo / 'repairs'
for old in repair.glob('q14-marking-q02.part*.b64'):
    old.unlink()
parts = sorted((repair / 'q14-clean').glob('part*'))
assert parts, 'Quiz 14 repair archive parts missing'
(repair / 'q14-marking-q02.part0.b64').write_bytes(b''.join(p.read_bytes() for p in parts))

# Generates Quiz 18-95 from PDF pairs, saves all 95 answer rows,
# and checks every generated WEBP. It does not change the GitHub baseline.
auto_cropper.main()
manifest = json.loads((repo / 'quiz-data.json').read_text(encoding='utf-8'))
assert (manifest['quizCount'], manifest['questionCount']) == (95, 475)
assert len(manifest['answers']) == 95

# Independently re-read every original marking PDF to confirm the answers.
for quiz in range(18, 96):
    question_pdf = repo / 'incoming' / f'2028 QUIZ {quiz}.pdf'
    marking_pdf = repo / 'incoming' / f'2028 QUIZ {quiz} MARKING.pdf'
    assert question_pdf.is_file() and marking_pdf.is_file(), f'Missing PDF pair for Quiz {quiz}'
    with fitz.open(marking_pdf) as doc:
        starts = cropper.find_question_starts(doc, anchor_to_answers=True)
        official_answers = cropper.extract_answers(doc, starts)
    assert manifest['answers'][quiz - 1] == official_answers, (
        f'Quiz {quiz:02d}: manifest answers differ from marking PDF'
    )
    print(f'Quiz {quiz:02d}: marking answers match')

count = 0
for quiz in range(1, 96):
    for kind in ('questions', 'markings'):
        for number in range(1, 6):
            path = repo / 'daily_assets' / kind / f'quiz-{quiz:02d}' / f'q-{number:02d}.webp'
            assert path.is_file() and path.stat().st_size > 1000, path
            with Image.open(path) as image:
                image.load()  # decode actual pixels, not just the header
                assert image.width > 200 and image.height > 30, path
                assert ImageStat.Stat(image.convert('L')).stddev[0] >= 3, (
                    f'Blank or almost blank crop: {path}'
                )
            count += 1
assert count == 950
print('All 950 WEBP images decode with meaningful pixel content; PDF answers verified.')


In [ ]:
os.environ['REPAIR_FROM']='27'
os.environ['REPAIR_TO']='95'
subprocess.run([sys.executable,'scripts/repair_drive_existing.py'],cwd=repo,check=True)
print('Drive repairs uploaded and verified. Existing valid files remain untouched.')

In [ ]:
from urllib.request import urlopen
with urlopen('https://ict-daily-quiz-2028-nimesha.vercel.app/quiz-data.json',timeout=30) as r:
    public_data=json.load(r)
assert (public_data['quizCount'],public_data['questionCount'])==(95,475)
print('Production website verified: 95 quizzes / 475 questions.')